In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║                       ENGENHARIA DE ATRIBUTOS                                ║
# ║      Geração, Documentação e Auditoria de Features — PLD Nordeste            ║
# ╚══════════════════════════════════════════════════════════════════════════════╝
#
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  IDENTIFICAÇÃO
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  Autor       : Yago
#  Instituição : Universidade Federal do Ceará
#  Curso       : Engenharia Elétrica
#  Versão      : 3.0
#
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  DESCRIÇÃO / FUNÇÃO DO SCRIPT
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  Este script implementa a etapa de engenharia de atributos da arquitetura do
#  TCC, aplicada ao submercado Nordeste (PLD NORDESTE — Cenários A e B).
#  Ele unifica duas responsabilidades complementares em um único pipeline:
#
#    (a) GERAÇÃO DE FEATURES (PLDFeatureEngineer + FeatureEngineeringEngine):
#        Cria e persiste os atributos derivados a partir dos dados tratados
#        (passo 7 — tratamento de outliers), organizados em 6 grupos temáticos:
#        Mercado, Calendário, Intradiário, CMO, Físicas e Energia.
#        Garantia anti-leakage rigorosa: todos os parâmetros (quantis, médias,
#        z-scores, top-N features) são calculados EXCLUSIVAMENTE no treino.
#
#    (b) DOCUMENTAÇÃO E AUDITORIA (FeatureDocumentator):
#        Constrói automaticamente o dicionário de dados das features geradas,
#        classificando cada variável por grupo, tipo, método de geração, cálculo,
#        variáveis-base e uso (preditor, encoding do alvo, alvo-derivada).
#        Gera 7 gráficos diagnósticos e exporta o dicionário em CSV e Excel.
#
#  Ao final da execução, o script gera por cenário + estratégia:
#    (a) arquivos Parquet com features de treino e teste;
#    (b) parâmetros do fit serializados em JSON;
#    (c) relatórios CSV por cenário e consolidado;
#    (d) dicionário de features em CSV e Excel;
#    (e) 7 gráficos diagnósticos em PNG.
#
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  ENTRADAS (INPUTS)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  O script consome os arquivos gerados pela etapa de tratamento de outliers.
#  Os parâmetros de controle estão definidos nas constantes abaixo:
#
#    INPUT_DIR  : BASE_DIR / "7_tratamento_outliers"
#    SCENARIOS  : lista de cenários a processar (A e B)
#    STRATEGIES : lista de estratégias de seleção (padrão: HYBRID_AGGRESSIVE)
#
#  Estrutura esperada de entrada (por cenário + estratégia):
#    INPUT_DIR / {cenario} / treino / {strategy} / TREINO_CLEAN.parquet
#    INPUT_DIR / {cenario} / teste  / {strategy} / TESTE_CLEAN.parquet
#
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  SAÍDAS (OUTPUTS)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  Por cenário + estratégia em SAVE_DIR / {cenario} /:
#
#  1. treino/{strategy}/TREINO_FEATURES.parquet   — features do conjunto de treino
#  2. teste/{strategy}/TESTE_FEATURES.parquet     — features do conjunto de teste
#  3. parametros_fit/{strategy}/feature_params.json
#                                                 — parâmetros do fit (anti-leakage)
#  4. relatorios/relatorio_feature_engineering.csv — resumo do cenário
#
#  Artefatos de documentação (raiz de SAVE_DIR):
#  5. relatorio_consolidado_feature_engineering.csv
#  6. dicionario_features.csv  — dicionário completo de todas as features
#  7. dicionario_features.xlsx — mesma informação em Excel (requer openpyxl)
#  8. figuras/
#     ├── fig_1_features_por_grupo.png
#     ├── fig_2_tipos_variaveis_barras.png
#     ├── fig_3_metodos_geracao.png
#     ├── fig_4_uso_features.png
#     ├── fig_5_variaveis_base.png
#     ├── fig_6_composicao_dataset.png
#     └── fig_7_kpi_dashboard.png
#
#  Saída no terminal (Rich):
#     • Painel de boas-vindas com resumo de versão e correções
#     • Painel de FIT com confirmação de anti-leakage
#     • Tabela de resumo de features por grupo após cada transform
#     • Tabela consolidada de todos os cenários e splits processados
#     • Painel de encerramento com KPIs e tempo total
#
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  GRUPOS DE FEATURES GERADAS (resumo)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  N°  Grupo           Features principais
#  01  MERCADO         VOL_ANUAL/MENSAL, REGIME_VOL/PRECO/MENSAL, SPIKE, IQR, MEDIA_ANUAL
#  02  CALENDÁRIO      Feriados BR, véspera/pós-feriado, codificação cíclica (mês, dia)
#                      TRIMESTRE, SEMESTRE, QUINZENA, SEMANA_ANO, DIST_FERIADO
#  03  INTRADIÁRIO     PERIODO_DIA, HORARIO_PICO, HORA_SEN/COS, médias hora×dia×mês
#                      (target encoding fitado no treino), POS_RELATIVA_DIA
#  04  CMO             Spreads entre submercados, log/vol/quadrático por coluna CMO
#                      CMO_MEDIA/STD/MAX/MIN/RANGE_SUBMERCADOS, REGIME_CMO
#  05  FÍSICAS         C(TOP_N,2)=190 interações multiplicativas, x², log(x)
#                      TOP_N=20 features por correlação com o target (fitadas no treino)
#  06  ENERGIA         PRESSAO_SISTEMA, DEFICIT_ENERGIA, RATIO_GERACAO, EXCEDENTE_LOG
#                      EAR_ZSCORE/PERCENTIL/ALERTA_ESCASSEZ, ENA_ZSCORE, LOG_ENA, EAR×CMO
#
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  CONSIDERAÇÕES INICIAIS E OBSERVAÇÕES TÉCNICAS
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#
#  1. AMBIENTE DE EXECUÇÃO
#     O script foi desenvolvido e testado no Google Colab (Python 3.10+).
#     Pode ser executado localmente, mas requer ajuste manual de BASE_DIR.
#     Antes de executar, montar o Google Drive:
#
#         from google.colab import drive
#         drive.mount('/content/drive')
#
#  2. PRÉ-REQUISITO
#     Este script depende dos arquivos .parquet gerados pela etapa de tratamento
#     de outliers (passo 7). Certifique-se de que essa etapa foi concluída com
#     sucesso antes de executar este script.
#
#  3. ARQUITETURA ANTI-LEAKAGE
#     Nenhuma feature usa PLD(t) no cálculo — sem lags nem rolling.
#     Todos os parâmetros do fit (quantis, médias horárias, z-scores, correlações,
#     top-N features) são calculados SOMENTE no treino e aplicados ao teste via
#     FeatureParams.save() / FeatureParams.load(). O teste nunca influencia
#     nenhuma decisão de fit.
#
#  4. CORREÇÕES APLICADAS (v3.0)
#     • TOP_N_FEATURES: 40 → 20 (C(40,2)=780 → C(20,2)=190 interações)
#     • Grupo 5 (_grupo_fisicas): colunas inseridas diretamente em df em loop
#       com del imediato do array — elimina pd.concat de DataFrame gigante.
#     • Grupo 4 (_grupo_cmo): idem — sem acúmulo em dict intermediário.
#     • Amostra de correlação: 8.000 → 5.000 linhas.
#     • gc.collect() adicionado ao final de cada grupo.
#     • Cast float32 aplicado inline, sem cópia intermediária.
#     • transform() descarta coluna DATA antes do Grupo 5 para liberar RAM.
#     • Documentação integrada ao mesmo script (era script separado).
#
#  5. GERENCIAMENTO DE MEMÓRIA
#     O Grupo 5 (Físicas) é o mais custoso em RAM: C(20,2)=190 arrays float32
#     + 20 quadráticos + 20 logarítmicos. Cada coluna é inserida diretamente
#     no DataFrame e o array temporário deletado imediatamente (del arr).
#     gc.collect() é chamado entre todos os grupos. _log_mem() imprime o uso
#     de RAM em cada checkpoint para acompanhamento no Colab.
#
#  6. DOCUMENTAÇÃO AUTOMÁTICA (FeatureDocumentator)
#     Funciona em dois modos:
#       REAL    : lê o feature_params.json e o parquet gerado para classificar
#                 as colunas reais produzidas pelo pipeline.
#       TEÓRICO : se os arquivos não existirem, reconstrói o catálogo
#                 teoricamente a partir das constantes do script.
#     O catálogo NAMED contém a ficha técnica de todas as features nomeadas
#     (fórmula, método, tipo, variáveis-base, uso). Features dinâmicas
#     (interações, spreads, log_*, *_2) são classificadas por padrão de nome.
#
#  7. FERIADOS
#     São considerados apenas feriados nacionais fixos (data fixa anual).
#     Feriados móveis (Carnaval, Corpus Christi, Sexta-Feira Santa) não estão
#     incluídos nesta versão e podem ser adicionados em FERIADOS_FIXOS.
#
#  8. REPRODUTIBILIDADE
#     Os parâmetros exatos do fit são persistidos em feature_params.json com
#     todos os dicionários de médias horárias, quantis e nomes das base_cols.
#     A flag reload_params=True reutiliza o JSON existente sem refitar,
#     garantindo reprodutibilidade total em reexecuções parciais.
#
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  DEPENDÊNCIAS
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  Biblioteca        Versão mínima  Finalidade
#  numpy             1.23           Operações numéricas e geração de features
#  pandas            1.5            Manipulação de DataFrames e Parquet
#  matplotlib        3.5            Geração dos 7 gráficos diagnósticos
#  rich              13.0           Tabelas, painéis e progresso no terminal
#  openpyxl          3.0            Exportação do dicionário em .xlsx (opcional)
#  json              —              Serialização dos parâmetros do fit
#  gc                —              Gerenciamento explícito de memória
#
#  Instalação rápida (Google Colab / pip):
#      !pip install rich matplotlib openpyxl
#  (numpy, pandas, gc e json são stdlib ou já disponíveis no Colab por padrão)
#
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  ESTRUTURA DE DIRETÓRIOS GERADA
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  8_engenharia_atributos/
#  ├── cenario_A_todos_anos/
#  │   ├── treino/HYBRID_AGGRESSIVE/TREINO_FEATURES.parquet
#  │   ├── teste/HYBRID_AGGRESSIVE/TESTE_FEATURES.parquet
#  │   ├── parametros_fit/HYBRID_AGGRESSIVE/feature_params.json
#  │   └── relatorios/relatorio_feature_engineering.csv
#  ├── cenario_B_sem_2022_2023/
#  │   └── (mesma estrutura)
#  ├── relatorio_consolidado_feature_engineering.csv
#  ├── dicionario_features.csv
#  ├── dicionario_features.xlsx
#  └── figuras/
#      ├── fig_1_features_por_grupo.png
#      ├── fig_2_tipos_variaveis_barras.png
#      ├── fig_3_metodos_geracao.png
#      ├── fig_4_uso_features.png
#      ├── fig_5_variaveis_base.png
#      ├── fig_6_composicao_dataset.png
#      └── fig_7_kpi_dashboard.png
#
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  COMO EXECUTAR (Google Colab)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  Célula 1 — Montar o Drive:
#      from google.colab import drive
#      drive.mount('/content/drive')
#
#  Célula 2 — Instalar dependências:
#      !pip install rich matplotlib openpyxl
#
#  Célula 3 — Executar o script:
#      exec(open('engenharia_atributos_pld_nordeste.py').read())
#   ou simplesmente executar este arquivo como módulo principal.
#
# ══════════════════════════════════════════════════════════════════════════════

import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

import gc
import json
import os
import time
from dataclasses import dataclass, field
from itertools import combinations
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from rich.console import Console
from rich.panel import Panel
from rich.table import Table
from rich.progress import Progress, SpinnerColumn, TextColumn, BarColumn, TimeElapsedColumn
from rich.theme import Theme
from rich import box


# ==============================================================================
# 🎨 TEMA VISUAL
# ==============================================================================
custom_theme = Theme({
    "info"      : "bold cyan",
    "warning"   : "bold yellow",
    "error"     : "bold red",
    "success"   : "bold green",
    "header"    : "bold white on blue",
    "metric"    : "bold magenta",
    "dim"       : "dim white",
    "treino"    : "bold steel_blue1",
    "teste"     : "bold orange1",
    "cenario_a" : "bold green",
    "cenario_b" : "bold magenta",
})
console = Console(theme=custom_theme)

# Paleta de cores para os gráficos — fundo branco e tons de azul marinho
C = {
    "bg"         : "#FFFFFF",
    "panel"      : "#FFFFFF",
    "grid"       : "#E0E0E0",
    "text"       : "#333333",
    "sub"        : "#666666",
    "navy_dark"  : "#001529",
    "navy"       : "#002B5B",
    "navy_light" : "#00509E",
    "blue_soft"  : "#4C9EEB",
    "gray_light" : "#B0C4DE",
    "gray_dark"  : "#808080",
}

# Cor por grupo nos gráficos
COR_GRUPO = {
    "1-MERCADO"    : C["navy_dark"],
    "2-CALENDÁRIO" : C["navy"],
    "3-INTRADIÁRIO": C["navy_light"],
    "4-CMO"        : "#1A5F7A",
    "5-FÍSICAS"    : "#227C9D",
    "6-ENERGIA"    : C["blue_soft"],
    "ORIGINAL"     : C["gray_light"],
    "ESTRUTURAL"   : C["gray_dark"],
}


# ==============================================================================
# ⚙️ CONFIGURAÇÕES DO AMBIENTE
# ==============================================================================
BASE_DIR  = Path("/content/drive/MyDrive/TCC_PLD_Project/09-ESCRITA_TCC/PARTES_TCC/codigos/dados")
INPUT_DIR = BASE_DIR / "7_tratamento_outliers"
SAVE_DIR  = BASE_DIR / "8_engenharia_atributos"

# ── Cenários a processar ──────────────────────────────────────────────────────
SCENARIOS: List[Dict] = [
    {"label": "cenario_A_todos_anos",    "desc": "FIT com todos os anos"},
    {"label": "cenario_B_sem_2022_2023", "desc": "FIT excluindo 2022 e 2023"},
]

# ── Estratégias de seleção de variáveis ──────────────────────────────────────
STRATEGIES: List[str] = ["HYBRID_AGGRESSIVE"]

# ── Coluna-alvo (TARGET) do submercado Nordeste ───────────────────────────────
TARGET_CANONICAL: str       = "01_CCEE_NORDESTE_TARGET_PLD_HORA_NORDESTE"
_TARGET_TOKENS  : List[str] = ["nordeste", "target"]

# FIX v3.0: reduzido de 40 → 20 para cortar interações de C(40,2)=780 → C(20,2)=190
TOP_N_FEATURES: int = 20

# ── Constantes de domínio ─────────────────────────────────────────────────────
# Feriados nacionais fixos (mês, dia) — móveis não incluídos nesta versão
FERIADOS_FIXOS: List[Tuple[int, int]] = [
    (1, 1), (4, 21), (5, 1), (9, 7),
    (10, 12), (11, 2), (11, 15), (12, 25),
]

# Horas de pico de demanda elétrica (ONS — período de ponta)
HORAS_PICO: List[int] = list(range(18, 22))

# Limiares de regime de volatilidade (anual, mensal e horária)
VOL_ANUAL_LIMIARES  = [-np.inf, 1_000,  20_000, np.inf]
VOL_MENSAL_LIMIARES = [-np.inf, 500,    5_000,  np.inf]
VOL_HORA_LIMIARES   = [-np.inf, 50,     500,    np.inf]


# ==============================================================================
# 🎯 DETECÇÃO DINÂMICA DE TARGET
# ==============================================================================
def detectar_target(df: pd.DataFrame) -> str:
    """
    Localiza a coluna-alvo (TARGET) no DataFrame em quatro níveis de fallback:
      1. Nome exato (TARGET_CANONICAL)
      2. Tokens ["nordeste", "target"] simultâneos no nome da coluna
      3. Tokens ["target", "nordeste"] (ordem alternativa)
      4. Tokens ["pld", "nordeste"]
    Levanta KeyError com diagnóstico se nenhuma correspondência for encontrada.
    """
    if TARGET_CANONICAL in df.columns:
        console.print(f"   [success]✔  TARGET (nome exato): [bold]{TARGET_CANONICAL}[/][/]")
        return TARGET_CANONICAL

    cols_lower = {c: c.lower() for c in df.columns}

    for tokens in [_TARGET_TOKENS, ["target", "nordeste"], ["pld", "nordeste"]]:
        cands = [c for c, cl in cols_lower.items() if all(t in cl for t in tokens)]
        if cands:
            console.print(f"   [warning]⚠  TARGET via {tokens}: [bold]{cands[0]}[/][/]")
            return cands[0]

    raise KeyError(
        f"Coluna TARGET não encontrada.\n"
        f"Nome canônico esperado: {TARGET_CANONICAL!r}\n"
        f"Colunas disponíveis (primeiras 30): {list(df.columns[:30])}"
    )


# ==============================================================================
# ⏱️ UTILITÁRIO DE TEMPO
# ==============================================================================
class TimeTracker:
    """Context manager para cronometrar estágios e exibir o tempo no terminal."""
    def __init__(self, label: str):
        self.label  = label
        self._start = 0.0

    def __enter__(self) -> "TimeTracker":
        self._start = time.perf_counter()
        return self

    def __exit__(self, *_) -> None:
        elapsed = time.perf_counter() - self._start
        console.print(f"   [dim]⏱ {self.label}: {elapsed:.2f}s[/]")


# ==============================================================================
# 📋 DATACLASS — PARÂMETROS DO FIT (anti-leakage)
# ==============================================================================
@dataclass
class FeatureParams:
    """
    Contêiner de todos os parâmetros calculados no treino e aplicados ao teste.
    Serializado em JSON para garantir reprodutibilidade total em reexecuções.
    """
    target_col         : str   = ""
    target_q1          : float = 0.0
    target_q3          : float = 0.0
    target_p1          : float = 0.0
    target_p99         : float = 0.0

    # Perfis intradiários (target encoding fitado no treino)
    media_hora         : Dict[int, float]             = field(default_factory=dict)
    vol_hora           : Dict[int, float]             = field(default_factory=dict)
    media_hora_dia     : Dict[Tuple[int, int], float] = field(default_factory=dict)
    media_hora_mes     : Dict[Tuple[int, int], float] = field(default_factory=dict)

    # Parâmetros do CMO
    cmo_media_q1       : float = 0.0
    cmo_media_q3       : float = 0.0
    cmo_sudeste_col    : str   = ""

    # Top-N features por correlação com o target
    base_cols          : List[str] = field(default_factory=list)

    # Parâmetros de energia (z-score fitado no treino)
    earm_mean          : float = 0.0
    earm_std           : float = 1.0
    earm_q30           : float = 0.0
    ena_mean           : float = 0.0
    ena_std            : float = 1.0

    def save(self, path: Path) -> None:
        """Serializa os parâmetros em JSON com chaves convertidas para string."""
        path.parent.mkdir(parents=True, exist_ok=True)
        payload = {
            "target_col"      : self.target_col,
            "target_q1"       : self.target_q1,
            "target_q3"       : self.target_q3,
            "target_p1"       : self.target_p1,
            "target_p99"      : self.target_p99,
            "media_hora"      : {str(k): v for k, v in self.media_hora.items()},
            "vol_hora"        : {str(k): v for k, v in self.vol_hora.items()},
            "media_hora_dia"  : {f"{k[0]}_{k[1]}": v for k, v in self.media_hora_dia.items()},
            "media_hora_mes"  : {f"{k[0]}_{k[1]}": v for k, v in self.media_hora_mes.items()},
            "cmo_media_q1"    : self.cmo_media_q1,
            "cmo_media_q3"    : self.cmo_media_q3,
            "cmo_sudeste_col" : self.cmo_sudeste_col,
            "base_cols"       : self.base_cols,
            "earm_mean"       : self.earm_mean,
            "earm_std"        : self.earm_std,
            "earm_q30"        : self.earm_q30,
            "ena_mean"        : self.ena_mean,
            "ena_std"         : self.ena_std,
        }
        with open(path, "w", encoding="utf-8") as f:
            json.dump(payload, f, ensure_ascii=False, indent=2)
        console.print(f"   [success]✔  Parâmetros do FIT salvos:[/] [dim]{path}[/]")

    @classmethod
    def load(cls, path: Path) -> "FeatureParams":
        """Desserializa os parâmetros do JSON, reconstruindo chaves de tupla."""
        with open(path, encoding="utf-8") as f:
            d = json.load(f)
        obj = cls()
        obj.target_col    = d.get("target_col", "")
        obj.target_q1     = d["target_q1"]
        obj.target_q3     = d["target_q3"]
        obj.target_p1     = d["target_p1"]
        obj.target_p99    = d["target_p99"]
        obj.media_hora    = {int(k): v for k, v in d["media_hora"].items()}
        obj.vol_hora      = {int(k): v for k, v in d["vol_hora"].items()}
        obj.media_hora_dia = {
            (int(k.split("_")[0]), int(k.split("_")[1])): v
            for k, v in d["media_hora_dia"].items()
        }
        obj.media_hora_mes = {
            (int(k.split("_")[0]), int(k.split("_")[1])): v
            for k, v in d["media_hora_mes"].items()
        }
        obj.cmo_media_q1    = d["cmo_media_q1"]
        obj.cmo_media_q3    = d["cmo_media_q3"]
        obj.cmo_sudeste_col = d.get("cmo_sudeste_col", "")
        obj.base_cols       = d["base_cols"]
        obj.earm_mean       = d["earm_mean"]
        obj.earm_std        = d["earm_std"]
        obj.earm_q30        = d["earm_q30"]
        obj.ena_mean        = d["ena_mean"]
        obj.ena_std         = d["ena_std"]
        console.print(f"   [info]Parâmetros do FIT carregados:[/] [dim]{path}[/]")
        return obj


# ==============================================================================
# 🔧 UTILITÁRIOS DE LEITURA & PADRONIZAÇÃO
# ==============================================================================

def _extract_key_int(df: pd.DataFrame, col: str,
                     fallback: int = 1,
                     lower: int = 1,
                     upper: int = 9999) -> pd.Series:
    """
    Extrai coluna KEY_* como inteiro com três estratégias em cascata:
      1. Coluna já numérica — converte diretamente
      2. String conversível — pd.to_numeric com coerce
      3. String de data — parse de datetime e extração do componente
    Aplica clip nos limites e substitui NaN pelo fallback.
    """
    raw = df[col]
    if pd.api.types.is_numeric_dtype(raw):
        return raw.fillna(fallback).clip(lower=lower, upper=upper).astype(int)

    converted = pd.to_numeric(raw, errors="coerce")
    if converted.notna().mean() > 0.5:
        return converted.fillna(fallback).clip(lower=lower, upper=upper).astype(int)

    console.print(f"   [warning]⚠  {col!r} parece string de data — tentando parse de datetime…[/]")
    try:
        parsed = pd.to_datetime(raw, errors="coerce")
        component_map = {
            "KEY_ANO" : parsed.dt.year,
            "KEY_MES" : parsed.dt.month,
            "KEY_DIA" : parsed.dt.day,
            "KEY_HORA": parsed.dt.hour,
        }
        result = component_map.get(col, parsed.dt.year)
        return result.fillna(fallback).clip(lower=lower, upper=upper).astype(int)
    except Exception:
        console.print(f"   [error]✘  Falha ao parsear {col!r}. Usando fallback={fallback}.[/]")
        return pd.Series(fallback, index=df.index, dtype=int)


def _build_datetime_safe(df: pd.DataFrame) -> pd.Series:
    """
    Constrói uma Series de pd.Timestamp a partir das colunas KEY_ANO/MES/DIA/HORA.
    Usa _extract_key_int com limites de validação para cada componente.
    """
    ano  = _extract_key_int(df, "KEY_ANO",  fallback=2000, lower=1900, upper=2100)
    mes  = _extract_key_int(df, "KEY_MES",  fallback=1,    lower=1,    upper=12)
    dia  = _extract_key_int(df, "KEY_DIA",  fallback=1,    lower=1,    upper=31)
    hora = _extract_key_int(df, "KEY_HORA", fallback=0,    lower=0,    upper=23)
    return pd.to_datetime({"year": ano, "month": mes, "day": dia, "hour": hora},
                          errors="coerce")


def _drop_string_date_cols(df: pd.DataFrame) -> pd.DataFrame:
    """
    Remove colunas que contêm datas como strings (ex.: "2021-01-01 00:00:00").
    Identifica candidatas por nome (prefixo/sufixo DATA) ou por amostragem
    do conteúdo (string com 4 dígitos seguidos de '-').
    Preserva colunas com prefixo KEY_ mesmo que o nome contenha "DATA".
    """
    cols_to_drop = []
    for col in df.columns:
        col_upper = col.upper()
        if (col_upper.startswith("DATA") or col_upper.endswith("DATA")) \
                and not col_upper.startswith("KEY_"):
            cols_to_drop.append(col)
            continue
        if df[col].dtype == object and not col_upper.startswith("KEY_"):
            sample = df[col].dropna().head(5)
            if len(sample) > 0:
                first = str(sample.iloc[0])
                if len(first) >= 7 and first[:4].isdigit() and first[4] == "-":
                    cols_to_drop.append(col)

    if cols_to_drop:
        console.print(
            f"   [warning]⚠  Removendo {len(cols_to_drop)} coluna(s) string-data: "
            f"{cols_to_drop[:5]}{'…' if len(cols_to_drop) > 5 else ''}[/]"
        )
        df = df.drop(columns=cols_to_drop, errors="ignore")
    return df


def _log_mem(df: pd.DataFrame, label: str = "") -> None:
    """Imprime uso de memória do DataFrame (MB) e número de colunas."""
    mb = df.memory_usage(deep=True).sum() / 1e6
    console.print(f"   [dim]💾 RAM df {label}: {mb:.1f} MB | colunas: {df.shape[1]}[/]")


# ==============================================================================
# 🏗️ ENGINE DE FEATURE ENGINEERING
# ==============================================================================
class PLDFeatureEngineer:
    """
    Engenheiro de Atributos para Previsão Horária do PLD Nordeste.

    Garantias anti-leakage:
      • Sem lags/rolling — nenhuma feature usa PLD(t) no cálculo.
      • Todos os parâmetros (quantis, médias, correlações) fitados SOMENTE no treino.
      • TOP_N_FEATURES=20 → C(20,2)=190 interações (v3.0 reduzido de 40).
      • Grupo 5 insere colunas diretamente no df sem pd.concat (fix RAM).
      • gc.collect() entre todos os grupos com checkpoint de memória.
    """

    def __init__(self):
        self.target  : Optional[str]           = None
        self.params  : Optional[FeatureParams] = None
        self.counters: Dict[str, int]          = {}

    # ──────────────────────────────────────────────────────────────────────────
    # 🔓 FIT — calcula parâmetros EXCLUSIVAMENTE no treino
    # ──────────────────────────────────────────────────────────────────────────
    def fit(self, df_treino: pd.DataFrame) -> "PLDFeatureEngineer":
        """
        Fita todos os parâmetros de geração de features usando apenas o treino.
        Ordem: quantis do target → perfis intradiários → CMO → top-N features
        por correlação (amostra de 5.000 linhas) → parâmetros de energia.
        """
        console.print(Panel(
            "[treino]FIT — parâmetros calculados APENAS no TREINO[/]\n"
            "[dim]Estes valores serão aplicados ao TESTE (anti-leakage)[/]",
            title="[header] FIT DE PARÂMETROS [/]",
            border_style="steel_blue1",
            padding=(0, 2),
        ))

        self.target = detectar_target(df_treino)
        t = self.target
        p = FeatureParams()
        p.target_col = t

        # ── Quantis do target ──────────────────────────────────────────────────
        p.target_q1  = float(df_treino[t].quantile(0.25))
        p.target_q3  = float(df_treino[t].quantile(0.75))
        p.target_p1  = float(df_treino[t].quantile(0.01))
        p.target_p99 = float(df_treino[t].quantile(0.99))

        # ── Perfis intradiários (target encoding fitado no treino) ─────────────
        dt         = _build_datetime_safe(df_treino)
        hora_s     = dt.dt.hour
        dia_s      = dt.dt.weekday
        mes_s      = dt.dt.month

        p.media_hora     = df_treino.groupby(hora_s)[t].mean().to_dict()
        p.vol_hora       = df_treino.groupby(hora_s)[t].std().to_dict()
        p.media_hora_dia = df_treino.groupby([hora_s, dia_s])[t].mean().to_dict()
        p.media_hora_mes = df_treino.groupby([hora_s, mes_s])[t].mean().to_dict()

        # ── Parâmetros do CMO ─────────────────────────────────────────────────
        cmo_cols = [c for c in df_treino.columns if "cmo" in c.lower()]
        if cmo_cols:
            cmo_media_tmp  = df_treino[cmo_cols].mean(axis=1)
            p.cmo_media_q1 = float(cmo_media_tmp.quantile(0.25))
            p.cmo_media_q3 = float(cmo_media_tmp.quantile(0.75))
            del cmo_media_tmp
            cmo_sudeste = next(
                (c for c in cmo_cols
                 if "sudeste" in c.lower() or "_se_" in c.lower() or c.lower().endswith("_se")),
                None
            )
            p.cmo_sudeste_col = cmo_sudeste or ""

        # ── Top-N features por correlação com o target (amostra 5k) ───────────
        exclude_tokens = [t.lower(), "key_", "data"]
        numeric_cols   = [
            c for c in df_treino.columns
            if not any(tok in c.lower() for tok in exclude_tokens)
            and df_treino[c].dtype.kind in ("f", "i", "u")
            and c != t
        ]
        console.print(
            f"   [info]🔎 Correlação em {len(numeric_cols):,} colunas (amostra 5k)…[/]"
        )
        _SAMPLE_N = min(5_000, len(df_treino))
        df_sample = df_treino[numeric_cols + [t]].sample(n=_SAMPLE_N, random_state=42)
        mat       = np.nan_to_num(df_sample.values.astype(np.float32), nan=0.0)
        y         = mat[:, -1];   X    = mat[:, :-1]
        y_c       = y - y.mean(); X_c  = X - X.mean(axis=0)
        y_std     = float(y_c.std()) + 1e-9
        X_std     = X_c.std(axis=0) + 1e-9
        corr_abs  = np.abs((X_c * y_c[:, None]).mean(axis=0) / (X_std * y_std))
        top_idx   = np.argsort(corr_abs)[::-1][:TOP_N_FEATURES]
        p.base_cols = [numeric_cols[i] for i in top_idx]
        console.print(f"   [info]🔥 Top {len(p.base_cols)} features selecionadas.[/]")
        del df_sample, mat, X, y, X_c, y_c, corr_abs
        gc.collect()

        # ── Parâmetros de energia (z-score fitado no treino) ──────────────────
        earm_col = next((c for c in df_treino.columns
                         if "earm" in c.lower() or "ear" in c.lower()), None)
        ena_col  = next((c for c in df_treino.columns if "ena" in c.lower()), None)
        if earm_col:
            p.earm_mean = float(df_treino[earm_col].mean())
            p.earm_std  = float(df_treino[earm_col].std() + 1e-6)
            p.earm_q30  = float(df_treino[earm_col].quantile(0.30))
        if ena_col:
            p.ena_mean  = float(df_treino[ena_col].mean())
            p.ena_std   = float(df_treino[ena_col].std() + 1e-6)

        self.params = p
        console.print("   [success]✔  FIT concluído.[/]")
        return self

    # ──────────────────────────────────────────────────────────────────────────
    # 🔓 TRANSFORM — aplica parâmetros do fit (treino ou teste)
    # ──────────────────────────────────────────────────────────────────────────
    def transform(self, df: pd.DataFrame, split_label: str = "TREINO") -> pd.DataFrame:
        """
        Aplica todos os grupos de features ao DataFrame usando exclusivamente
        os parâmetros fitados no treino. Fluxo:
          cast float32 → drop string-data → build DATA → G1 a G4 →
          drop DATA → G5 (mais pesado) → G6.
        gc.collect() e _log_mem() entre cada grupo.
        """
        if self.params is None:
            raise RuntimeError("Chame .fit(df_treino) antes de .transform().")

        if self.params.target_col and self.params.target_col in df.columns:
            self.target = self.params.target_col
        else:
            console.print(f"   [warning]⚠  target_col não encontrado em {split_label}. Redetectando…[/]")
            self.target = detectar_target(df)

        cor = "treino" if split_label == "TREINO" else "teste"
        console.print(Panel(
            f"[{cor}]{split_label}[/] — TARGET=[bold]{self.target}[/bold] | "
            f"{len(self.params.base_cols)} base_cols\n"
            "[dim]Parâmetros do FIT (treino) aplicados sem recalcular[/]",
            title=f"[header] TRANSFORM — {split_label} [/]",
            border_style="steel_blue1" if split_label == "TREINO" else "orange1",
            padding=(0, 2),
        ))

        self.counters = {}

        with TimeTracker("Cast float64 → float32"):
            df = self._cast_float32(df)
        with TimeTracker("Drop colunas string-data"):
            df = _drop_string_date_cols(df)
        _log_mem(df, "após drops iniciais")

        with TimeTracker("Criação da coluna DATA"):
            df = self._build_datetime(df)

        with TimeTracker("Grupo 1 — Mercado / Regime"):
            df = self._grupo_mercado(df)
        gc.collect();  _log_mem(df, "pós-G1")

        with TimeTracker("Grupo 2 — Calendário & Feriados"):
            df = self._grupo_calendario(df)
        gc.collect();  _log_mem(df, "pós-G2")

        with TimeTracker("Grupo 3 — Curva Intradiária"):
            df = self._grupo_intradiario(df)
        gc.collect();  _log_mem(df, "pós-G3")

        with TimeTracker("Grupo 4 — CMO"):
            df = self._grupo_cmo(df)
        gc.collect();  _log_mem(df, "pós-G4")

        # Drop DATA antes do Grupo 5 — libera RAM antes da etapa mais pesada
        df = df.drop(columns=["DATA"], errors="ignore")
        gc.collect()

        with TimeTracker("Grupo 5 — Físicas (interações)"):
            df = self._grupo_fisicas(df)
        gc.collect();  _log_mem(df, "pós-G5")

        with TimeTracker("Grupo 6 — Energia"):
            df = self._grupo_energia(df)
        gc.collect();  _log_mem(df, "pós-G6 (final)")

        self._print_summary(split_label)
        return df

    # ──────────────────────────────────────────────────────────────────────────
    # 🔒 UTILITÁRIOS
    # ──────────────────────────────────────────────────────────────────────────

    def _cast_float32(self, df: pd.DataFrame) -> pd.DataFrame:
        """Converte todas as colunas float64 para float32 in-place (sem cópia)."""
        float64_cols = df.select_dtypes(include=[np.float64]).columns.tolist()
        if float64_cols:
            df[float64_cols] = df[float64_cols].astype(np.float32)
        return df

    def _build_datetime(self, df: pd.DataFrame) -> pd.DataFrame:
        """Adiciona coluna DATA como pd.Timestamp a partir das KEY_* temporais."""
        df["DATA"] = _build_datetime_safe(df)
        return df

    # ──────────────────────────────────────────────────────────────────────────
    # 🔒 GRUPO 1 — MERCADO / REGIME
    # ──────────────────────────────────────────────────────────────────────────
    def _grupo_mercado(self, df: pd.DataFrame) -> pd.DataFrame:
        """
        Features de regime de mercado baseadas no target:
          VOL_ANUAL/MENSAL: variância do PLD por ano/mês (encoding de alvo)
          REGIME_VOL/PRECO/MENSAL: binning por limiares ou quantis do treino
          SPIKE_ALTO/BAIXO/SPIKE: flags de outlier por P99/P1 do treino
          IQR_MENSAL, MEDIA_ANUAL: estatísticas mensais/anuais do target
        """
        before = len(df.columns)
        t = self.target;  p = self.params
        ano_num = _extract_key_int(df, "KEY_ANO", fallback=2000, lower=1900, upper=2100)
        mes_num = _extract_key_int(df, "KEY_MES", fallback=1,    lower=1,    upper=12)

        df["VOL_ANUAL"]     = df.groupby(ano_num)[t].transform("var").astype(np.float32)
        df["REGIME_VOL"]    = pd.cut(df["VOL_ANUAL"], VOL_ANUAL_LIMIARES, labels=[0,1,2]).astype(np.float32)
        df["REGIME_PRECO"]  = pd.cut(df[t], [-np.inf, p.target_q1, p.target_q3, np.inf], labels=[0,1,2]).astype(np.float32)
        df["SPIKE_ALTO"]    = (df[t] >= p.target_p99).astype(np.int8)
        df["SPIKE_BAIXO"]   = (df[t] <= p.target_p1).astype(np.int8)
        df["SPIKE"]         = ((df["SPIKE_ALTO"] == 1) | (df["SPIKE_BAIXO"] == 1)).astype(np.int8)
        df["VOL_MENSAL"]    = df.groupby([ano_num, mes_num])[t].transform("var").astype(np.float32)
        df["REGIME_MENSAL"] = pd.cut(df["VOL_MENSAL"], VOL_MENSAL_LIMIARES, labels=[0,1,2]).astype(np.float32)
        df["IQR_MENSAL"]    = (
            df.groupby([ano_num, mes_num])[t]
            .transform(lambda x: x.quantile(0.75) - x.quantile(0.25))
            .astype(np.float32)
        )
        df["MEDIA_ANUAL"]   = df.groupby(ano_num)[t].transform("mean").astype(np.float32)

        self.counters["mercado"] = len(df.columns) - before
        return df

    # ──────────────────────────────────────────────────────────────────────────
    # 🔒 GRUPO 2 — CALENDÁRIO & FERIADOS
    # ──────────────────────────────────────────────────────────────────────────
    def _grupo_calendario(self, df: pd.DataFrame) -> pd.DataFrame:
        """
        Features de calendário e feriados nacionais:
          DIA_SEMANA, IS_FIM_DE_SEMANA, IS_FERIADO, IS_DIA_UTIL, TIPO_DIA
          VESP_FERIADO, POS_FERIADO, FERIADO_PROLONGADO, DIST_FERIADO (horas)
          Codificações cíclicas: MES_SEN/COS, DIA_SEN/COS
          TRIMESTRE, SEMESTRE, QUINZENA, SEMANA_ANO
        """
        before = len(df.columns)
        dt = df["DATA"]
        df["DIA_SEMANA"]       = dt.dt.weekday.astype(np.int8)
        df["IS_FIM_DE_SEMANA"] = df["DIA_SEMANA"].isin([5, 6]).astype(np.int8)

        feriado_set = {
            pd.Timestamp(year=y, month=m, day=d)
            for y in _extract_key_int(df, "KEY_ANO", fallback=2020, lower=1900, upper=2100).unique()
            for m, d in FERIADOS_FIXOS
        }
        datas_dia                = dt.dt.normalize()
        df["IS_FERIADO"]         = datas_dia.isin(feriado_set).astype(np.int8)
        df["IS_DIA_UTIL"]        = ((df["IS_FIM_DE_SEMANA"] == 0) & (df["IS_FERIADO"] == 0)).astype(np.int8)
        df["TIPO_DIA"]           = np.where(df["IS_FERIADO"]==1, 2, np.where(df["IS_FIM_DE_SEMANA"]==1, 1, 0)).astype(np.int8)
        prox_dia                 = datas_dia + pd.Timedelta(days=1)
        ant_dia                  = datas_dia - pd.Timedelta(days=1)
        df["VESP_FERIADO"]       = prox_dia.isin(feriado_set).astype(np.int8)
        df["POS_FERIADO"]        = ant_dia.isin(feriado_set).astype(np.int8)
        df["FERIADO_PROLONGADO"] = (
            (df["IS_FERIADO"] | df["VESP_FERIADO"] | df["POS_FERIADO"]).astype(bool) &
            df["IS_FIM_DE_SEMANA"].astype(bool)
        ).astype(np.int8)

        # Distância temporal até o próximo feriado (em horas) via searchsorted
        feriado_sorted = np.array(sorted(feriado_set), dtype="datetime64[ns]")
        dt_vals        = dt.values.astype("datetime64[ns]")
        idx            = np.searchsorted(feriado_sorted, dt_vals)
        valid          = idx < len(feriado_sorted)
        dist_h         = np.where(
            valid,
            (feriado_sorted[np.clip(idx, 0, len(feriado_sorted) - 1)].astype(np.int64)
             - dt_vals.astype(np.int64)) / 3_600_000_000_000,
            np.nan,
        )
        df["DIST_FERIADO"] = dist_h.astype(np.float32)
        df["MES_SEN"]      = np.sin(2 * np.pi * dt.dt.month / 12).astype(np.float32)
        df["MES_COS"]      = np.cos(2 * np.pi * dt.dt.month / 12).astype(np.float32)
        df["DIA_SEN"]      = np.sin(2 * np.pi * df["DIA_SEMANA"] / 7).astype(np.float32)
        df["DIA_COS"]      = np.cos(2 * np.pi * df["DIA_SEMANA"] / 7).astype(np.float32)
        df["TRIMESTRE"]    = dt.dt.quarter.astype(np.int8)
        df["SEMESTRE"]     = ((dt.dt.month - 1) // 6 + 1).astype(np.int8)
        df["QUINZENA"]     = (dt.dt.day > 15).astype(np.int8)
        df["SEMANA_ANO"]   = dt.dt.isocalendar().week.astype(np.int16)

        self.counters["calendario"] = len(df.columns) - before
        return df

    # ──────────────────────────────────────────────────────────────────────────
    # 🔒 GRUPO 3 — CURVA INTRADIÁRIA
    # ──────────────────────────────────────────────────────────────────────────
    def _grupo_intradiario(self, df: pd.DataFrame) -> pd.DataFrame:
        """
        Features de comportamento intradiário do PLD:
          HORA, PERIODO_DIA, HORARIO_PICO, HORA_SEN/COS (cíclicas)
          MEDIA_HORA, VOL_HORA, REGIME_HORA: target encoding fitado no treino
          MEDIA_HORA_DIA, MEDIA_HORA_MES: cruzamentos hora×dia e hora×mês
          POS_RELATIVA_DIA: HORA / 23 (posição normalizada no dia)
        """
        before = len(df.columns)
        p = self.params;  dt = df["DATA"]
        df["HORA"]           = dt.dt.hour.astype(np.int8)
        df["PERIODO_DIA"]    = pd.cut(df["HORA"], bins=[-1,5,11,17,23], labels=[0,1,2,3]).astype(np.int8)
        df["HORARIO_PICO"]   = df["HORA"].isin(HORAS_PICO).astype(np.int8)
        df["HORA_SEN"]       = np.sin(2 * np.pi * df["HORA"] / 24).astype(np.float32)
        df["HORA_COS"]       = np.cos(2 * np.pi * df["HORA"] / 24).astype(np.float32)
        df["MEDIA_HORA"]     = df["HORA"].map(p.media_hora).astype(np.float32)
        df["VOL_HORA"]       = df["HORA"].map(p.vol_hora).astype(np.float32)
        df["REGIME_HORA"]    = pd.cut(df["VOL_HORA"], VOL_HORA_LIMIARES, labels=[0,1,2]).astype(np.float32)
        dia_s = dt.dt.weekday;  mes_s = dt.dt.month
        df["MEDIA_HORA_DIA"] = list(zip(df["HORA"], dia_s))
        df["MEDIA_HORA_DIA"] = df["MEDIA_HORA_DIA"].map(p.media_hora_dia).astype(np.float32)
        df["MEDIA_HORA_MES"] = list(zip(df["HORA"], mes_s))
        df["MEDIA_HORA_MES"] = df["MEDIA_HORA_MES"].map(p.media_hora_mes).astype(np.float32)
        df["POS_RELATIVA_DIA"] = (df["HORA"] / 23).astype(np.float32)

        self.counters["intradiario"] = len(df.columns) - before
        return df

    # ──────────────────────────────────────────────────────────────────────────
    # 🔒 GRUPO 4 — CMO
    # ──────────────────────────────────────────────────────────────────────────
    def _grupo_cmo(self, df: pd.DataFrame) -> pd.DataFrame:
        """
        Features derivadas das colunas de Custo Marginal de Operação (CMO):
          Spreads entre pares de submercados (C(n_cmo,2) combinações)
          Estatísticas de linha: MEDIA/STD/MAX/MIN/RANGE_SUBMERCADOS
          Por coluna CMO: LOG_*, VOL_MENSAL_*, *_2 (quadrático)
          REGIME_CMO: regime de custo marginal com quantis do treino
        Inserção direta no df (sem acúmulo em dict) — FIX RAM v3.0.
        """
        before   = len(df.columns)
        p        = self.params
        cmo_cols = [c for c in df.columns if "cmo" in c.lower()]

        if not cmo_cols:
            console.print("   [warning]⚠ Nenhuma coluna CMO encontrada. Grupo 4 ignorado.[/]")
            self.counters["cmo"] = 0
            return df

        console.print(f"   [info]📡 CMO columns: {cmo_cols}[/]")

        # Spreads entre pares — inserção direta para evitar pd.concat
        if len(cmo_cols) >= 2:
            for a, b in combinations(cmo_cols, 2):
                col_name = (
                    f"SPREAD_"
                    f"{a.replace('FEATURE_','').replace('ccee_pld_','').replace('01_CCEE_','')}_"
                    f"{b.replace('FEATURE_','').replace('ccee_pld_','').replace('01_CCEE_','')}"
                )
                df[col_name] = (df[a] - df[b]).astype(np.float32)

        df["CMO_MEDIA_SUBMERCADOS"] = df[cmo_cols].mean(axis=1).astype(np.float32)
        df["CMO_STD_SUBMERCADOS"]   = df[cmo_cols].std(axis=1).astype(np.float32)
        df["CMO_MAX_SUBMERCADOS"]   = df[cmo_cols].max(axis=1).astype(np.float32)
        df["CMO_MIN_SUBMERCADOS"]   = df[cmo_cols].min(axis=1).astype(np.float32)
        df["CMO_RANGE_SUBMERCADOS"] = (df["CMO_MAX_SUBMERCADOS"] - df["CMO_MIN_SUBMERCADOS"]).astype(np.float32)

        _ano_cmo = _extract_key_int(df, "KEY_ANO", fallback=2000, lower=1900, upper=2100)
        _mes_cmo = _extract_key_int(df, "KEY_MES", fallback=1,    lower=1,    upper=12)

        for col in cmo_cols:
            base = (col.replace("FEATURE_VAL_","").replace("FEATURE_","")
                       .replace("ccee_pld_","").replace("01_CCEE_",""))
            df[f"LOG_{base}"]        = np.log1p(df[col].clip(lower=0)).astype(np.float32)
            df[f"VOL_MENSAL_{base}"] = df.groupby([_ano_cmo, _mes_cmo])[col].transform("var").astype(np.float32)
            df[f"{base}_2"]          = (df[col].values ** 2).astype(np.float32)

        df["REGIME_CMO"] = pd.cut(
            df["CMO_MEDIA_SUBMERCADOS"],
            [-np.inf, p.cmo_media_q1, p.cmo_media_q3, np.inf],
            labels=[0, 1, 2]
        ).astype(np.float32)

        self.counters["cmo"] = len(df.columns) - before
        return df

    # ──────────────────────────────────────────────────────────────────────────
    # 🔒 GRUPO 5 — FÍSICAS (interações + polinomiais)
    # ──────────────────────────────────────────────────────────────────────────
    def _grupo_fisicas(self, df: pd.DataFrame) -> pd.DataFrame:
        """
        Features de interação e transformação das top-N colunas base:
          C(TOP_N,2)=190 interações multiplicativas (a×b)
          TOP_N quadráticos (x²)
          TOP_N logarítmicos (log1p(x))

        FIX RAM v3.0 — PRINCIPAL CORREÇÃO:
          Antes: criava dict com ~860 arrays, depois pd.concat(df, df_phys)
                 → pico RAM = df + arrays + resultado = 3×
          Agora: insere cada coluna diretamente em df com del imediato do array
                 → pico RAM ≈ df + 1 array temporário ≈ 1.003×
        """
        p = self.params
        OUTPUT_COLS = {"SPIKE_ALTO", "SPIKE_BAIXO", "SPIKE", "REGIME_PRECO"}
        base_cols   = [
            c for c in p.base_cols
            if c in df.columns
            and c not in OUTPUT_COLS
            and df[c].dtype.kind in ("f", "i", "u")
        ]

        if not base_cols:
            console.print("   [warning]⚠ base_cols vazio. Grupo 5 ignorado.[/]")
            self.counters["fisicas"] = 0
            return df

        n_interact = len(list(combinations(base_cols, 2)))
        console.print(
            f"   [info]⚛️  {n_interact} interações + {len(base_cols)} pol. "
            f"+ {len(base_cols)} log — inserção direta (sem concat)[/]"
        )

        count = 0
        for a, b in combinations(base_cols, 2):      # interações a×b
            arr = (df[a].values * df[b].values).astype(np.float32)
            df[f"{a}_x_{b}"] = arr;  del arr;  count += 1

        for col in base_cols:                         # quadráticos e logarítmicos
            arr2 = (df[col].values ** 2).astype(np.float32)
            df[f"{col}_2"] = arr2;  del arr2;  count += 1
            arr_log = np.log1p(df[col].values.clip(min=0)).astype(np.float32)
            df[f"log_{col}"] = arr_log;  del arr_log;  count += 1

        self.counters["fisicas"] = count
        return df

    # ──────────────────────────────────────────────────────────────────────────
    # 🔒 GRUPO 6 — ENERGIA
    # ──────────────────────────────────────────────────────────────────────────
    def _grupo_energia(self, df: pd.DataFrame) -> pd.DataFrame:
        """
        Features de balanço e escassez energética:
          PRESSAO_SISTEMA, DEFICIT_ENERGIA, RATIO_GERACAO, EXCEDENTE_LOG
          EAR_ZSCORE, EAR_PERCENTIL, ALERTA_ESCASSEZ (z-score e limiares do treino)
          ENA_ZSCORE, LOG_ENA, EAR_x_CMO (interação energia armazenada × custo)
        """
        before = len(df.columns)
        p = self.params

        carga_col   = next((c for c in df.columns
                            if "carga" in c.lower() and "nordeste" in c.lower()), None)
        if not carga_col:
            carga_col = next((c for c in df.columns if "carga" in c.lower()), None)
        geracao_col = next((c for c in df.columns
                            if ("geracao" in c.lower() or "ger_" in c.lower())
                            and "nordeste" in c.lower()), None)
        if not geracao_col:
            geracao_col = next((c for c in df.columns
                                if "geracao" in c.lower() or "ger_" in c.lower()), None)
        earm_col = next((c for c in df.columns
                         if "earm" in c.lower() or "ear" in c.lower()), None)
        ena_col  = next((c for c in df.columns if "ena" in c.lower()), None)

        if carga_col and geracao_col:
            df["PRESSAO_SISTEMA"] = (df[carga_col] / (df[geracao_col] + 1)).astype(np.float32)
            df["DEFICIT_ENERGIA"] = (df[carga_col] - df[geracao_col]).astype(np.float32)
            df["RATIO_GERACAO"]   = (df[geracao_col] / (df[carga_col] + 1e-6)).astype(np.float32)
            df["EXCEDENTE_LOG"]   = np.log1p((df[geracao_col] - df[carga_col]).clip(lower=0)).astype(np.float32)

        if earm_col:
            df["EAR_ZSCORE"]      = ((df[earm_col] - p.earm_mean) / p.earm_std).astype(np.float32)
            df["EAR_PERCENTIL"]   = df[earm_col].rank(pct=True).astype(np.float32)
            df["ALERTA_ESCASSEZ"] = (df[earm_col] < p.earm_q30).astype(np.int8)

        if ena_col:
            df["ENA_ZSCORE"] = ((df[ena_col] - p.ena_mean) / p.ena_std).astype(np.float32)
            df["LOG_ENA"]    = np.log1p(df[ena_col].clip(lower=0)).astype(np.float32)

        if earm_col and "CMO_MEDIA_SUBMERCADOS" in df.columns:
            df["EAR_x_CMO"] = (df[earm_col].values * df["CMO_MEDIA_SUBMERCADOS"].values).astype(np.float32)

        self.counters["energia"] = len(df.columns) - before
        return df

    # ──────────────────────────────────────────────────────────────────────────
    # 🔒 RESUMO RICH
    # ──────────────────────────────────────────────────────────────────────────
    def _print_summary(self, split_label: str = "") -> None:
        """Exibe tabela Rich com contagem de features geradas por grupo."""
        table = Table(
            title=f"📊 Resumo de Features — {split_label}",
            box=box.ROUNDED, show_header=True,
            header_style="bold white on dark_blue",
        )
        table.add_column("Grupo",         style="cyan",  justify="left")
        table.add_column("Novas Colunas", style="green", justify="center")
        table.add_column("Descrição",     style="white", justify="left")

        descricoes = {
            "mercado"    : "Vol anual/mensal, regimes, SPIKE, IQR, MEDIA_ANUAL",
            "calendario" : "Feriados BR, véspera, pós-feriado, codif. cíclica, trimestre, semana",
            "intradiario": "Períodos, HORA_SEN/COS, médias hora×dia×mês (treino), pos_relativa",
            "cmo"        : "Spreads, log, vol, regime CMO — inserção direta v3.0",
            "fisicas"    : f"C({TOP_N_FEATURES},2) interações + col² + log(col) — inserção direta v3.0",
            "energia"    : "Pressão, déficit, EAR z/pct/alerta, ENA, EAR×CMO",
        }

        total = 0
        for grupo, count in self.counters.items():
            table.add_row(grupo.upper(), str(count), descricoes.get(grupo, "—"))
            total += count
        table.add_row("[bold]TOTAL[/]", f"[bold green]{total}[/]", "[bold]Features geradas[/]")
        console.print(table)


# ==============================================================================
# 📖 DOCUMENTADOR DE FEATURES
# ==============================================================================
class FeatureDocumentator:
    """
    Gera automaticamente o dicionário de dados das features produzidas pelo
    PLDFeatureEngineer. Classifica cada coluna por grupo, tipo, método de
    geração, fórmula de cálculo, variáveis-base e uso (anti-leakage).

    Funciona em dois modos:
      REAL    : lê feature_params.json e o parquet de features para classificar
                as colunas reais produzidas pelo pipeline.
      TEÓRICO : reconstrói o catálogo teoricamente a partir das constantes do
                script (usado quando os arquivos ainda não existem).
    """

    # ── Catálogo de features nomeadas — ficha técnica completa ────────────────
    _T = "PLD"
    NAMED: Dict = {
        "VOL_ANUAL":          ("1-MERCADO",     "Contínua",           "Agregação groupby+var",         f"Var({_T}) por ano",                           "KEY_ANO, alvo",                        "Encoding do alvo (agregado)"),
        "REGIME_VOL":         ("1-MERCADO",     "Ordinal {0,1,2}",    "Binning (pd.cut)",              "Faixa de VOL_ANUAL em [1e3, 2e4]",             "VOL_ANUAL",                            "Preditor"),
        "REGIME_PRECO":       ("1-MERCADO",     "Ordinal {0,1,2}",    "Binning por quantis (FIT)",     f"Faixa de {_T} em [Q1, Q3] do treino",         "alvo",                                 "Alvo-derivada (rótulo)"),
        "SPIKE_ALTO":         ("1-MERCADO",     "Binária 0/1",        "Limiar (FIT)",                  f"{_T} ≥ P99(treino)",                          "alvo",                                 "Alvo-derivada (rótulo)"),
        "SPIKE_BAIXO":        ("1-MERCADO",     "Binária 0/1",        "Limiar (FIT)",                  f"{_T} ≤ P1(treino)",                           "alvo",                                 "Alvo-derivada (rótulo)"),
        "SPIKE":              ("1-MERCADO",     "Binária 0/1",        "Lógica OR",                     "SPIKE_ALTO ∨ SPIKE_BAIXO",                     "SPIKE_ALTO, SPIKE_BAIXO",              "Alvo-derivada (rótulo)"),
        "VOL_MENSAL":         ("1-MERCADO",     "Contínua",           "Agregação groupby+var",         f"Var({_T}) por (ano, mês)",                    "KEY_ANO, KEY_MES, alvo",               "Encoding do alvo (agregado)"),
        "REGIME_MENSAL":      ("1-MERCADO",     "Ordinal {0,1,2}",    "Binning (pd.cut)",              "Faixa de VOL_MENSAL em [500, 5e3]",            "VOL_MENSAL",                           "Preditor"),
        "IQR_MENSAL":         ("1-MERCADO",     "Contínua",           "Agregação groupby+IQR",         f"Q3−Q1 de {_T} por (ano, mês)",                "KEY_ANO, KEY_MES, alvo",               "Encoding do alvo (agregado)"),
        "MEDIA_ANUAL":        ("1-MERCADO",     "Contínua",           "Agregação groupby+mean",        f"Média de {_T} por ano",                       "KEY_ANO, alvo",                        "Encoding do alvo (agregado)"),
        "DIA_SEMANA":         ("2-CALENDÁRIO",  "Inteiro 0–6",        "Extração de datetime",          "weekday(DATA)",                                "DATA",                                 "Preditor"),
        "IS_FIM_DE_SEMANA":   ("2-CALENDÁRIO",  "Binária 0/1",        "isin",                          "DIA_SEMANA ∈ {5,6}",                           "DIA_SEMANA",                           "Preditor"),
        "IS_FERIADO":         ("2-CALENDÁRIO",  "Binária 0/1",        "Lookup de feriados (isin)",     "DATA ∈ feriados BR fixos",                     "DATA",                                 "Preditor"),
        "IS_DIA_UTIL":        ("2-CALENDÁRIO",  "Binária 0/1",        "Lógica AND",                    "¬fim de semana ∧ ¬feriado",                    "IS_FIM_DE_SEMANA, IS_FERIADO",         "Preditor"),
        "TIPO_DIA":           ("2-CALENDÁRIO",  "Ordinal {0,1,2}",    "np.where aninhado",             "0=útil, 1=FDS, 2=feriado",                     "IS_FERIADO, IS_FIM_DE_SEMANA",         "Preditor"),
        "VESP_FERIADO":       ("2-CALENDÁRIO",  "Binária 0/1",        "Lookup de feriados (isin)",     "(DATA+1 dia) ∈ feriados",                      "DATA",                                 "Preditor"),
        "POS_FERIADO":        ("2-CALENDÁRIO",  "Binária 0/1",        "Lookup de feriados (isin)",     "(DATA−1 dia) ∈ feriados",                      "DATA",                                 "Preditor"),
        "FERIADO_PROLONGADO": ("2-CALENDÁRIO",  "Binária 0/1",        "Lógica AND/OR",                 "(feriado∨vésp∨pós) ∧ FDS",                     "IS_FERIADO, VESP_FERIADO, POS_FERIADO, IS_FIM_DE_SEMANA", "Preditor"),
        "DIST_FERIADO":       ("2-CALENDÁRIO",  "Contínua (h)",       "Distância temporal (searchsorted)", "Horas até o próximo feriado",              "DATA",                                 "Preditor"),
        "MES_SEN":            ("2-CALENDÁRIO",  "Cíclica",            "Codificação cíclica",           "sin(2π·mês/12)",                               "DATA",                                 "Preditor"),
        "MES_COS":            ("2-CALENDÁRIO",  "Cíclica",            "Codificação cíclica",           "cos(2π·mês/12)",                               "DATA",                                 "Preditor"),
        "DIA_SEN":            ("2-CALENDÁRIO",  "Cíclica",            "Codificação cíclica",           "sin(2π·dia_sem/7)",                            "DIA_SEMANA",                           "Preditor"),
        "DIA_COS":            ("2-CALENDÁRIO",  "Cíclica",            "Codificação cíclica",           "cos(2π·dia_sem/7)",                            "DIA_SEMANA",                           "Preditor"),
        "TRIMESTRE":          ("2-CALENDÁRIO",  "Ordinal 1–4",        "Extração de datetime",          "quarter(DATA)",                                "DATA",                                 "Preditor"),
        "SEMESTRE":           ("2-CALENDÁRIO",  "Ordinal 1–2",        "Aritmética de datetime",        "(mês−1)//6 + 1",                               "DATA",                                 "Preditor"),
        "QUINZENA":           ("2-CALENDÁRIO",  "Binária 0/1",        "Limiar",                        "dia > 15",                                     "DATA",                                 "Preditor"),
        "SEMANA_ANO":         ("2-CALENDÁRIO",  "Inteiro 1–53",       "Extração ISO",                  "isocalendar().week",                           "DATA",                                 "Preditor"),
        "HORA":               ("3-INTRADIÁRIO", "Inteiro 0–23",       "Extração de datetime",          "hour(DATA)",                                   "DATA",                                 "Preditor"),
        "PERIODO_DIA":        ("3-INTRADIÁRIO", "Ordinal {0..3}",     "Binning (pd.cut)",              "Madrugada/Manhã/Tarde/Noite",                  "HORA",                                 "Preditor"),
        "HORARIO_PICO":       ("3-INTRADIÁRIO", "Binária 0/1",        "isin",                          "HORA ∈ {18,19,20,21}",                         "HORA",                                 "Preditor"),
        "HORA_SEN":           ("3-INTRADIÁRIO", "Cíclica",            "Codificação cíclica",           "sin(2π·hora/24)",                              "HORA",                                 "Preditor"),
        "HORA_COS":           ("3-INTRADIÁRIO", "Cíclica",            "Codificação cíclica",           "cos(2π·hora/24)",                              "HORA",                                 "Preditor"),
        "MEDIA_HORA":         ("3-INTRADIÁRIO", "Contínua",           "Target encoding (FIT)",         f"Média de {_T} por hora (treino)",              "HORA, alvo",                           "Encoding do alvo (agregado)"),
        "VOL_HORA":           ("3-INTRADIÁRIO", "Contínua",           "Target encoding (FIT)",         f"Desvio de {_T} por hora (treino)",             "HORA, alvo",                           "Encoding do alvo (agregado)"),
        "REGIME_HORA":        ("3-INTRADIÁRIO", "Ordinal {0,1,2}",    "Binning (pd.cut)",              "Faixa de VOL_HORA em [50, 500]",               "VOL_HORA",                             "Preditor"),
        "MEDIA_HORA_DIA":     ("3-INTRADIÁRIO", "Contínua",           "Target encoding (FIT)",         f"Média de {_T} por (hora, dia_sem)",            "HORA, DIA_SEMANA, alvo",               "Encoding do alvo (agregado)"),
        "MEDIA_HORA_MES":     ("3-INTRADIÁRIO", "Contínua",           "Target encoding (FIT)",         f"Média de {_T} por (hora, mês)",                "HORA, mês, alvo",                      "Encoding do alvo (agregado)"),
        "POS_RELATIVA_DIA":   ("3-INTRADIÁRIO", "Contínua 0–1",       "Normalização",                  "HORA / 23",                                    "HORA",                                 "Preditor"),
        "CMO_MEDIA_SUBMERCADOS": ("4-CMO",      "Contínua",           "Estatística por linha",         "média das colunas CMO",                        "colunas CMO",                          "Preditor"),
        "CMO_STD_SUBMERCADOS":   ("4-CMO",      "Contínua",           "Estatística por linha",         "desvio das colunas CMO",                       "colunas CMO",                          "Preditor"),
        "CMO_MAX_SUBMERCADOS":   ("4-CMO",      "Contínua",           "Estatística por linha",         "máx das colunas CMO",                          "colunas CMO",                          "Preditor"),
        "CMO_MIN_SUBMERCADOS":   ("4-CMO",      "Contínua",           "Estatística por linha",         "mín das colunas CMO",                          "colunas CMO",                          "Preditor"),
        "CMO_RANGE_SUBMERCADOS": ("4-CMO",      "Contínua",           "Diferença",                     "CMO_MAX − CMO_MIN",                            "CMO_MAX_SUBMERCADOS, CMO_MIN_SUBMERCADOS", "Preditor"),
        "REGIME_CMO":            ("4-CMO",      "Ordinal {0,1,2}",    "Binning por quantis (FIT)",     "Faixa de CMO_MEDIA em [Q1,Q3] treino",         "CMO_MEDIA_SUBMERCADOS",                "Preditor"),
        "PRESSAO_SISTEMA":    ("6-ENERGIA",     "Contínua",           "Razão",                         "carga / (geração + 1)",                        "carga, geração",                       "Preditor"),
        "DEFICIT_ENERGIA":    ("6-ENERGIA",     "Contínua",           "Diferença",                     "carga − geração",                              "carga, geração",                       "Preditor"),
        "RATIO_GERACAO":      ("6-ENERGIA",     "Contínua",           "Razão",                         "geração / (carga + ε)",                        "carga, geração",                       "Preditor"),
        "EXCEDENTE_LOG":      ("6-ENERGIA",     "Contínua",           "Transformação log1p",           "log1p(max(geração−carga, 0))",                 "carga, geração",                       "Preditor"),
        "EAR_ZSCORE":         ("6-ENERGIA",     "Contínua",           "Z-score (FIT)",                 "(EAR − μ_treino) / σ_treino",                  "EAR (energia armazenada)",             "Preditor"),
        "EAR_PERCENTIL":      ("6-ENERGIA",     "Contínua 0–1",       "Rank percentil",                "rank percentual de EAR",                       "EAR",                                  "Preditor"),
        "ALERTA_ESCASSEZ":    ("6-ENERGIA",     "Binária 0/1",        "Limiar (FIT)",                  "EAR < Q30(treino)",                            "EAR",                                  "Preditor"),
        "ENA_ZSCORE":         ("6-ENERGIA",     "Contínua",           "Z-score (FIT)",                 "(ENA − μ_treino) / σ_treino",                  "ENA (afluência)",                      "Preditor"),
        "LOG_ENA":            ("6-ENERGIA",     "Contínua",           "Transformação log1p",           "log1p(max(ENA, 0))",                           "ENA",                                  "Preditor"),
        "EAR_x_CMO":          ("6-ENERGIA",     "Contínua",           "Interação multiplicativa",      "EAR × CMO_MEDIA_SUBMERCADOS",                  "EAR, CMO_MEDIA_SUBMERCADOS",           "Preditor"),
    }

    def __init__(self, save_dir: Path, params_json: Path, parquet_treino: Path):
        self.save_dir      = Path(save_dir)
        self.params_json   = Path(params_json)
        self.parquet_treino = Path(parquet_treino)

    def _classificar(self, nome: str, base_cols_set: set) -> dict:
        """
        Classifica uma coluna por grupo, tipo, método, fórmula, base e uso.
        Ordem de resolução: catálogo NAMED → padrões de nome dinâmicos
        (interações _x_, spreads SPREAD_, transformações LOG_/log_/*_2) →
        colunas estruturais (KEY_*, DATA) → colunas originais de entrada.
        """
        if nome in self.NAMED:
            g, t, m, calc, base, uso = self.NAMED[nome]
        elif "_x_" in nome and nome != "EAR_x_CMO":
            a, b = nome.split("_x_", 1)
            g, t, m = "5-FÍSICAS", "Contínua", "Interação multiplicativa (a×b)"
            calc, base, uso = f"{a} × {b}", f"{a}, {b}", "Preditor"
        elif nome.startswith("SPREAD_"):
            g, t, m = "4-CMO", "Contínua", "Diferença (spread a−b)"
            calc, base, uso = "CMO_a − CMO_b", "2 colunas CMO", "Preditor"
        elif nome.startswith("VOL_MENSAL_"):
            g, t, m = "4-CMO", "Contínua", "Agregação groupby+var"
            calc, base, uso = "Var(CMO) por (ano, mês)", nome.replace("VOL_MENSAL_", "CMO:"), "Preditor"
        elif nome.startswith("LOG_"):
            g, t, m = "4-CMO", "Contínua", "Transformação log1p"
            calc, base, uso = "log1p(CMO ≥ 0)", nome.replace("LOG_", "CMO:"), "Preditor"
        elif nome.startswith("log_"):
            g, t, m = "5-FÍSICAS", "Contínua", "Transformação log1p"
            calc, base, uso = "log1p(x ≥ 0)", nome[4:], "Preditor"
        elif nome.endswith("_2"):
            raiz = nome[:-2]
            if raiz in base_cols_set:
                g, t, m = "5-FÍSICAS", "Contínua", "Transformação polinomial (x²)"
                calc, base, uso = f"{raiz}²", raiz, "Preditor"
            else:
                g, t, m = "4-CMO", "Contínua", "Transformação polinomial (x²)"
                calc, base, uso = "CMO²", f"CMO: {raiz}", "Preditor"
        elif nome.startswith("CMO_"):
            g, t, m = "4-CMO", "Contínua", "Estatística por linha"
            calc, base, uso = "resumo das colunas CMO", "colunas CMO", "Preditor"
        elif nome.upper().startswith("KEY_") or nome.upper() in ("DATA",):
            g, t, m = "ESTRUTURAL", "Chave/tempo", "—"
            calc, base, uso = "chave de tempo (entrada)", "—", "Estrutural (chave/data)"
        else:
            g, t, m = "ORIGINAL", "Original", "—"
            calc, base, uso = "coluna de entrada (não engenheirada)", "—", "Original (entrada)"

        return dict(variavel=nome, grupo=g, tipo=t, metodo=m,
                    calculo=calc, variaveis_base=base, uso=uso)

    def _carregar_fontes(self):
        """
        Carrega base_cols do JSON e colunas reais do parquet quando disponíveis.
        Fallback para modo teórico (reconstrói catálogo pelas constantes do script).
        """
        base_cols, colunas, dtypes, modo = None, None, None, "TEÓRICO"

        if self.params_json.exists():
            try:
                with open(self.params_json, encoding="utf-8") as f:
                    base_cols = json.load(f).get("base_cols", None)
                console.print(f"   [success]✔  feature_params.json lido — {len(base_cols)} base_cols.[/]")
            except Exception as e:
                console.print(f"   [warning]⚠  Falha ao ler params: {e}[/]")

        if self.parquet_treino.exists():
            try:
                df_head = pd.read_parquet(self.parquet_treino).head(200)
                colunas = list(df_head.columns)
                dtypes  = {c: str(df_head[c].dtype) for c in colunas}
                modo    = "REAL (parquet do pipeline)"
                console.print(f"   [success]✔  Parquet lido — {len(colunas)} colunas reais.[/]")
            except Exception as e:
                console.print(f"   [warning]⚠  Falha ao ler parquet: {e}[/]")

        if colunas is None:
            console.print("   [info]ℹ  Modo TEÓRICO (padrões do script).[/]")
            if base_cols is None:
                base_cols = [f"base_{i:02d}" for i in range(1, TOP_N_FEATURES + 1)]
            colunas = list(self.NAMED.keys())
            for a, b in combinations(base_cols, 2):
                colunas.append(f"{a}_x_{b}")
            for c in base_cols:
                colunas += [f"{c}_2", f"log_{c}"]
            cmo_demo = ["CMO_SUDESTE", "CMO_SUL", "CMO_NORDESTE"]
            for a, b in combinations(cmo_demo, 2):
                colunas.append(f"SPREAD_{a}_{b}")
            for c in cmo_demo:
                colunas += [f"LOG_{c}", f"VOL_MENSAL_{c}", f"{c}_2"]
            colunas += ["KEY_ANO", "KEY_MES", "KEY_DIA", "KEY_HORA", TARGET_CANONICAL]

        if base_cols is None:
            base_cols = []

        return colunas, set(base_cols), base_cols, dtypes, modo

    def _construir_dicionario(self, colunas, base_cols_set, dtypes) -> pd.DataFrame:
        """Constrói o DataFrame do dicionário classificando todas as colunas."""
        linhas = [self._classificar(c, base_cols_set) for c in colunas]
        df = pd.DataFrame(linhas)
        df["dtype_real"] = df["variavel"].map(dtypes).fillna("—") if dtypes else "—"
        df.loc[df["variavel"].str.contains("TARGET", case=False), "uso"]   = "ALVO (y)"
        df.loc[df["variavel"].str.contains("TARGET", case=False), "grupo"] = "ESTRUTURAL"
        return df[["variavel","grupo","tipo","uso","metodo","calculo","variaveis_base","dtype_real"]]

    # ──────────────────────────────────────────────────────────────────────────
    # 📊 GRÁFICOS DIAGNÓSTICOS
    # ──────────────────────────────────────────────────────────────────────────

    @staticmethod
    def _estilo(ax):
        """Aplica estilo padrão (fundo branco, grid cinza, texto escuro) ao eixo."""
        ax.set_facecolor(C["panel"])
        ax.tick_params(colors=C["text"])
        for s in ax.spines.values():
            s.set_color(C["grid"])
        ax.xaxis.label.set_color(C["text"])
        ax.yaxis.label.set_color(C["text"])
        ax.title.set_color(C["text"])
        ax.grid(True, color=C["grid"], alpha=0.5, linestyle="--")

    @staticmethod
    def _salvar(fig, caminho: Path):
        """Salva figura com fundo branco e fecha o objeto para liberar memória."""
        fig.savefig(caminho, dpi=150, bbox_inches="tight", facecolor=fig.get_facecolor())
        plt.close(fig)
        console.print(f"   [success]🖼  Salvo: {caminho.name}[/]")

    def _gerar_graficos(self, df: pd.DataFrame, base_cols: list, fig_dir: Path) -> None:
        """
        Gera e salva os 7 gráficos diagnósticos do pipeline de features:
          1. Features por grupo (barras horizontais)
          2. Tipos de variáveis (barras horizontais com %)
          3. Métodos de geração (barras horizontais)
          4. Uso / anti-leakage (barras verticais)
          5. Variáveis-base das interações (barras horizontais)
          6. Composição do dataset final (barras verticais)
          7. Dashboard KPI (painel de texto estilizado)
        """
        fig_dir.mkdir(parents=True, exist_ok=True)
        eng = df[~df["grupo"].isin(["ORIGINAL", "ESTRUTURAL"])]

        # ── Fig 1: Features por grupo ──────────────────────────────────────────
        fig, ax = plt.subplots(figsize=(8, 6))
        fig.patch.set_facecolor(C["bg"]);  self._estilo(ax)
        porg = eng["grupo"].value_counts().sort_values()
        ax.barh(porg.index, porg.values,
                color=[COR_GRUPO.get(g, C["navy"]) for g in porg.index], edgecolor="white")
        for i, v in enumerate(porg.values):
            ax.text(v + max(porg.values)*0.01, i, f"{v}", va="center",
                    color=C["text"], fontsize=10, fontweight="bold")
        ax.set_title(f"Features Geradas por Grupo (Total = {len(eng)})", fontsize=14, pad=12)
        ax.set_xlabel("Nº de Variáveis")
        self._salvar(fig, fig_dir / "fig_1_features_por_grupo.png")

        # ── Fig 2: Tipos de variáveis ──────────────────────────────────────────
        fig, ax = plt.subplots(figsize=(10, 6))
        fig.patch.set_facecolor(C["bg"]);  self._estilo(ax)
        tipos = eng["tipo"].apply(lambda s: s.split()[0]).value_counts().sort_values()
        ax.barh(tipos.index, tipos.values, color=C["navy_light"], edgecolor="white")
        for i, v in enumerate(tipos.values):
            pct = v / len(eng) * 100
            ax.text(v + max(tipos.values)*0.01, i, f" {v}  ({pct:.1f}%)", va="center",
                    color=C["text"], fontsize=11, fontweight="bold")
        ax.set_title("Distribuição por Tipo de Variável", fontsize=14, pad=12)
        ax.set_xlabel("Nº de Variáveis")
        ax.set_xlim(0, max(tipos.values) * 1.15)
        self._salvar(fig, fig_dir / "fig_2_tipos_variaveis_barras.png")

        # ── Fig 3: Métodos de geração ──────────────────────────────────────────
        fig, ax = plt.subplots(figsize=(10, 6))
        fig.patch.set_facecolor(C["bg"]);  self._estilo(ax)
        met = eng["metodo"].value_counts().sort_values()
        ax.barh(met.index, met.values, color=C["navy"], edgecolor="white")
        for i, v in enumerate(met.values):
            ax.text(v + max(met.values)*0.01, i, f"{v}", va="center", color=C["text"], fontsize=9)
        ax.set_title("Métodos de Geração Aplicados", fontsize=14, pad=12)
        ax.set_xlabel("Nº de Variáveis")
        self._salvar(fig, fig_dir / "fig_3_metodos_geracao.png")

        # ── Fig 4: Uso / anti-leakage ──────────────────────────────────────────
        fig, ax = plt.subplots(figsize=(8, 6))
        fig.patch.set_facecolor(C["bg"]);  self._estilo(ax)
        uso = eng["uso"].value_counts()
        cores_uso = {"Preditor": C["navy"], "Encoding do alvo (agregado)": C["navy_light"],
                     "Alvo-derivada (rótulo)": C["blue_soft"]}
        ax.bar(range(len(uso)), uso.values,
               color=[cores_uso.get(u, C["navy"]) for u in uso.index], edgecolor="white")
        ax.set_xticks(range(len(uso)))
        ax.set_xticklabels(uso.index, rotation=15, ha="right", fontsize=9)
        for i, v in enumerate(uso.values):
            ax.text(i, v + max(uso.values)*0.01, f"{v}", ha="center", va="bottom",
                    color=C["text"], fontsize=11, fontweight="bold")
        ax.set_title("Uso das Features (Atenção ao Anti-Leakage)", fontsize=14, pad=12)
        ax.set_ylabel("Nº de Variáveis")
        self._salvar(fig, fig_dir / "fig_4_uso_features.png")

        # ── Fig 5: Variáveis-base das interações ───────────────────────────────
        fig, ax = plt.subplots(figsize=(8, 6))
        fig.patch.set_facecolor(C["bg"]);  self._estilo(ax)
        if base_cols:
            nomes  = [b if len(b) <= 26 else b[:24] + "…" for b in base_cols]
            filhas = len(base_cols) + 1
            ax.barh(range(len(nomes)), [filhas]*len(nomes), color=C["navy_dark"], edgecolor="white")
            ax.set_yticks(range(len(nomes)))
            ax.set_yticklabels(nomes, fontsize=8)
            ax.set_title(f"Variáveis-BASE das Interações (TOP-{len(base_cols)}) → {filhas} filhas cada",
                         fontsize=14, pad=12)
            ax.set_xlabel("Nº de Features Derivadas")
        else:
            ax.text(0.5, 0.5, "base_cols indisponível", ha="center", va="center",
                    color=C["sub"], transform=ax.transAxes)
            ax.set_title("Variáveis-BASE das Interações", fontsize=14, pad=12)
        self._salvar(fig, fig_dir / "fig_5_variaveis_base.png")

        # ── Fig 6: Composição do dataset final ────────────────────────────────
        fig, ax = plt.subplots(figsize=(7, 6))
        fig.patch.set_facecolor(C["bg"]);  self._estilo(ax)
        comp = df["grupo"].apply(
            lambda g: "Geradas" if g not in ("ORIGINAL","ESTRUTURAL")
                      else ("Originais" if g == "ORIGINAL" else "Estruturais")
        ).value_counts()
        ax.bar(range(len(comp)), comp.values,
               color=[C["navy"], C["gray_light"], C["gray_dark"]][:len(comp)], edgecolor="white")
        ax.set_xticks(range(len(comp)))
        ax.set_xticklabels(comp.index, fontsize=11)
        for i, v in enumerate(comp.values):
            ax.text(i, v + max(comp.values)*0.02, f"{v}", ha="center", va="bottom",
                    color=C["text"], fontsize=12, fontweight="bold")
        ax.set_title("Composição do Dataset Final", fontsize=14, pad=12)
        ax.set_ylabel("Nº de Colunas")
        ax.set_ylim(0, max(comp.values) * 1.1)
        self._salvar(fig, fig_dir / "fig_6_composicao_dataset.png")

        # ── Fig 7: Dashboard KPI ──────────────────────────────────────────────
        fig, ax = plt.subplots(figsize=(9, 4))
        fig.patch.set_facecolor(C["bg"]);  ax.set_facecolor(C["panel"]);  ax.axis("off")
        total_cols     = len(df)
        total_geradas  = len(eng)
        total_orig     = (df["grupo"] == "ORIGINAL").sum()
        dominante      = eng["grupo"].value_counts().idxmax()
        metodo_top     = eng["metodo"].value_counts().idxmax()
        ax.text(0.5, 0.9, "Dashboard de Feature Engineering (KPIs)",
                ha="center", va="center", fontsize=16, fontweight="bold", color=C["navy_dark"])
        bbox_kw = dict(facecolor=C["grid"], edgecolor="none", boxstyle="round,pad=1")
        ax.text(0.15, 0.6, f"{total_cols}\nColunas Totais",   ha="center", va="center",
                fontsize=14, color=C["navy"],       fontweight="bold", bbox=bbox_kw)
        ax.text(0.50, 0.6, f"{total_geradas}\nFeatures Geradas", ha="center", va="center",
                fontsize=14, color=C["navy_light"], fontweight="bold", bbox=bbox_kw)
        ax.text(0.85, 0.6, f"{total_orig}\nColunas Originais", ha="center", va="center",
                fontsize=14, color=C["sub"],        fontweight="bold", bbox=bbox_kw)
        ax.text(0.25, 0.25, f"Grupo Dominante:\n{dominante}",    ha="center", va="center",
                fontsize=11, color=C["text"])
        ax.text(0.75, 0.25, f"Método Mais Utilizado:\n{metodo_top}", ha="center", va="center",
                fontsize=11, color=C["text"])
        self._salvar(fig, fig_dir / "fig_7_kpi_dashboard.png")

    # ──────────────────────────────────────────────────────────────────────────
    # 🚀 ENTRY POINT — documentar
    # ──────────────────────────────────────────────────────────────────────────
    def documentar(self) -> pd.DataFrame:
        """
        Executa o fluxo completo de documentação:
          carrega fontes → constrói dicionário → exporta CSV/Excel →
          gera 7 gráficos → imprime resumo Rich no terminal.
        Retorna o DataFrame do dicionário de features.
        """
        console.print(Panel(
            "[info]Construindo dicionário de features do pipeline…[/]\n"
            f"[dim]Params JSON  : {self.params_json}[/]\n"
            f"[dim]Parquet treino: {self.parquet_treino}[/]",
            title="[header] 📖 DOCUMENTAÇÃO DE FEATURES [/header]",
            border_style="blue", padding=(0, 2),
        ))

        colunas, base_set, base_cols, dtypes, modo = self._carregar_fontes()
        df = self._construir_dicionario(colunas, base_set, dtypes)

        # ── Exportação do dicionário ───────────────────────────────────────────
        csv_out  = self.save_dir / "dicionario_features.csv"
        xlsx_out = self.save_dir / "dicionario_features.xlsx"
        df.to_csv(csv_out, index=False, encoding="utf-8-sig")
        console.print(f"   [success]✅ dicionario_features.csv salvo[/]")
        try:
            df.to_excel(xlsx_out, index=False)
            console.print(f"   [success]✅ dicionario_features.xlsx salvo[/]")
        except Exception:
            console.print("   [warning]⚠  xlsx requer openpyxl — CSV salvo com sucesso.[/]")

        # ── Gráficos diagnósticos ─────────────────────────────────────────────
        fig_dir = self.save_dir / "figuras"
        console.print(f"\n   [info]🖼  Gerando 7 gráficos em {fig_dir}…[/]")
        self._gerar_graficos(df, base_cols, fig_dir)

        # ── Resumo Rich no terminal ───────────────────────────────────────────
        eng = df[~df["grupo"].isin(["ORIGINAL", "ESTRUTURAL"])]
        tg = Table(title="Features por Grupo", box=box.ROUNDED, header_style="bold white on blue")
        tg.add_column("Grupo");  tg.add_column("Qtd", justify="right");  tg.add_column("Métodos principais")
        for g, sub in eng.groupby("grupo"):
            tg.add_row(g, str(len(sub)), ", ".join(sorted(sub["metodo"].unique())[:3]))
        console.print(tg)

        tt = Table(title="Features por Tipo", box=box.ROUNDED, header_style="bold white on blue")
        tt.add_column("Tipo");  tt.add_column("Qtd", justify="right")
        for tp, q in eng["tipo"].apply(lambda s: s.split()[0]).value_counts().items():
            tt.add_row(tp, str(q))
        console.print(tt)

        console.print(Panel(
            f"[success]Fonte da análise   : {modo}[/]\n"
            f"Total de colunas   : [bold]{len(df)}[/]\n"
            f"Features geradas   : [bold]{len(eng)}[/]\n"
            f"Colunas originais  : [bold]{(df['grupo']=='ORIGINAL').sum()}[/]\n"
            f"Variáveis-base     : [bold]{len(base_cols)}[/]\n"
            f"Interações C(N,2)  : [bold]{len(list(combinations(base_cols,2))) if base_cols else 0}[/]",
            title="[header] RESUMO — ENGENHARIA DE ATRIBUTOS (PLD NORDESTE) [/header]",
            border_style="green", padding=(0, 2),
        ))

        return df


# ==============================================================================
# 🚀 ENGINE PRINCIPAL — ORQUESTRADOR DO PIPELINE
# ==============================================================================
class FeatureEngineeringEngine:
    """
    Orquestrador do pipeline completo de engenharia de atributos.
    Itera sobre cenários e estratégias, executa fit/transform e,
    ao final, aciona o FeatureDocumentator para gerar o dicionário
    e os gráficos diagnósticos automaticamente.
    """

    def __init__(
        self,
        input_dir     : Path                = INPUT_DIR,
        save_dir      : Path                = SAVE_DIR,
        scenarios     : Optional[List[Dict]] = None,
        strategies    : Optional[List[str]]  = None,
        reload_params : bool                = False,
    ) -> None:
        self.input_dir     = Path(input_dir)
        self.save_dir      = Path(save_dir)
        self.scenarios     = scenarios or SCENARIOS
        self.strategies    = strategies or STRATEGIES
        self.reload_params = reload_params
        self._all_results  : List[Dict] = []
        self.save_dir.mkdir(parents=True, exist_ok=True)

    def run(self) -> None:
        """
        Ponto de entrada do pipeline. Exibe banner, itera sobre cenários,
        imprime relatório consolidado, salva CSV e executa a documentação.
        """
        self._print_banner()
        t_total = time.perf_counter()
        for scenario in self.scenarios:
            self._run_scenario(scenario)
        self._print_final_report(time.perf_counter() - t_total)
        self._salvar_relatorio_consolidado()
        self._executar_documentacao()

    def _run_scenario(self, scenario: Dict) -> None:
        """
        Processa um cenário completo: para cada estratégia, lê treino e teste,
        fita o PLDFeatureEngineer no treino, transforma treino e teste,
        salva os parquets e registra os resultados.
        """
        label  = scenario["label"]
        is_a   = "todos" in label
        cor    = "cenario_a" if is_a else "cenario_b"
        titulo = "CENÁRIO A — TODOS OS ANOS" if is_a else "CENÁRIO B — SEM 2022 E 2023"

        console.print()
        console.rule(f"[{cor}]{'='*20}  {titulo}  {'='*20}[/]")
        console.print(Panel(
            f"[{cor}]{scenario['desc']}[/]\n"
            f"[dim]Input : {self.input_dir / label}[/]\n"
            f"[dim]Output: {self.save_dir / label}[/]",
            title=f"[header] {titulo} [/]",
            border_style="green" if is_a else "magenta",
            padding=(0, 2),
        ))

        for strategy in self.strategies:
            console.rule(f"[header] {label} | ESTRATÉGIA: {strategy} [/]")

            treino_in  = self.input_dir / label / "treino" / strategy / "TREINO_CLEAN.parquet"
            teste_in   = self.input_dir / label / "teste"  / strategy / "TESTE_CLEAN.parquet"
            treino_out = self.save_dir  / label / "treino" / strategy / "TREINO_FEATURES.parquet"
            teste_out  = self.save_dir  / label / "teste"  / strategy / "TESTE_FEATURES.parquet"
            params_path = self.save_dir / label / "parametros_fit" / strategy / "feature_params.json"
            treino_out.parent.mkdir(parents=True, exist_ok=True)
            teste_out.parent.mkdir(parents=True, exist_ok=True)
            params_path.parent.mkdir(parents=True, exist_ok=True)

            if not treino_in.exists():
                console.print(f"   [error]✘  Treino não encontrado: {treino_in}[/]")
                self._all_results.append(self._erro("TREINO", strategy, label, "arquivo não encontrado"))
                self._all_results.append(self._erro("TESTE",  strategy, label, "arquivo não encontrado"))
                continue

            t0 = time.perf_counter()
            console.print(f"   [info]📂 Lendo TREINO: {treino_in.name}[/]")
            df_treino = _drop_string_date_cols(pd.read_parquet(treino_in))
            console.print(
                f"   [info]📐 Shape TREINO: {df_treino.shape} | "
                f"💾 {df_treino.memory_usage(deep=True).sum()/1e6:.1f} MB[/]"
            )

            engineer = PLDFeatureEngineer()

            if self.reload_params and params_path.exists():
                console.print(Panel(
                    f"[info]Reutilizando parâmetros do FIT:[/]\n[dim]{params_path}[/]",
                    title="[header] FIT (RELOAD) [/]", border_style="steel_blue1",
                ))
                engineer.params = FeatureParams.load(params_path)
                engineer.target = engineer.params.target_col
            else:
                with TimeTracker("FIT no TREINO"):
                    engineer.fit(df_treino)
                engineer.params.save(params_path)

            # ── TRANSFORM TREINO ──────────────────────────────────────────────
            try:
                with TimeTracker("TRANSFORM TREINO"):
                    df_treino = engineer.transform(df_treino, split_label="TREINO")
                console.print(
                    f"   [success]📐 Shape TREINO final: {df_treino.shape} | "
                    f"💾 {df_treino.memory_usage(deep=True).sum()/1e6:.1f} MB[/]"
                )
                with TimeTracker(f"Salvando TREINO → {treino_out.name}"):
                    df_treino.to_parquet(treino_out, index=False)
                console.print(f"   [success]✅ TREINO salvo: {treino_out}[/]")
                self._all_results.append({
                    "cenario"   : label,   "strategy"  : strategy,   "split"     : "TREINO",
                    "target_col": engineer.target,
                    "rows"      : len(df_treino),
                    "cols_out"  : df_treino.shape[1],
                    "elapsed_s" : round(time.perf_counter() - t0, 2),
                    "output_mb" : round(treino_out.stat().st_size / 1e6, 2),
                    "path"      : str(treino_out),
                    "status"    : "OK",
                })
            except Exception:
                console.print_exception()
                self._all_results.append(self._erro("TREINO", strategy, label))
            finally:
                del df_treino;  gc.collect()

            # ── TRANSFORM TESTE ───────────────────────────────────────────────
            if not teste_in.exists():
                console.print(f"   [warning]⚠  Teste não encontrado: {teste_in}[/]")
                self._all_results.append(self._erro("TESTE", strategy, label, "arquivo não encontrado"))
                continue

            t1 = time.perf_counter()
            console.print(f"   [info]📂 Lendo TESTE: {teste_in.name}[/]")
            df_teste = _drop_string_date_cols(pd.read_parquet(teste_in))
            console.print(
                f"   [info]📐 Shape TESTE: {df_teste.shape} | "
                f"💾 {df_teste.memory_usage(deep=True).sum()/1e6:.1f} MB[/]"
            )

            try:
                with TimeTracker("TRANSFORM TESTE (parâmetros do TREINO — anti-leakage)"):
                    df_teste = engineer.transform(df_teste, split_label="TESTE")
                console.print(
                    f"   [success]📐 Shape TESTE final: {df_teste.shape} | "
                    f"💾 {df_teste.memory_usage(deep=True).sum()/1e6:.1f} MB[/]"
                )
                with TimeTracker(f"Salvando TESTE → {teste_out.name}"):
                    df_teste.to_parquet(teste_out, index=False)
                console.print(f"   [success]✅ TESTE salvo: {teste_out}[/]")
                self._all_results.append({
                    "cenario"   : label,   "strategy"  : strategy,   "split"    : "TESTE",
                    "target_col": engineer.target,
                    "rows"      : len(df_teste),
                    "cols_out"  : df_teste.shape[1],
                    "elapsed_s" : round(time.perf_counter() - t1, 2),
                    "output_mb" : round(teste_out.stat().st_size / 1e6, 2),
                    "path"      : str(teste_out),
                    "status"    : "OK",
                })
            except Exception:
                console.print_exception()
                self._all_results.append(self._erro("TESTE", strategy, label))
            finally:
                del df_teste;  gc.collect()

        self._salvar_relatorio_cenario(
            [r for r in self._all_results if r.get("cenario") == label],
            self.save_dir / label / "relatorios",
        )

    def _executar_documentacao(self) -> None:
        """
        Aciona o FeatureDocumentator ao final do pipeline usando o primeiro
        cenário e estratégia bem-sucedidos para localizar os artefatos.
        """
        console.print()
        console.rule("[header] 📖 DOCUMENTAÇÃO AUTOMÁTICA DE FEATURES [/header]")

        # Usa o Cenário A por padrão para a documentação
        cenario_doc = self.scenarios[0]["label"]
        strategy_doc = self.strategies[0]
        params_json    = self.save_dir / cenario_doc / "parametros_fit" / strategy_doc / "feature_params.json"
        parquet_treino = self.save_dir / cenario_doc / "treino" / strategy_doc / "TREINO_FEATURES.parquet"

        doc = FeatureDocumentator(
            save_dir      = self.save_dir,
            params_json   = params_json,
            parquet_treino = parquet_treino,
        )
        doc.documentar()

    @staticmethod
    def _erro(split: str, strategy: str, cenario: str, motivo: str = "exceção") -> Dict:
        return {"cenario": cenario, "strategy": strategy, "split": split,
                "target_col": "—", "rows": 0, "cols_out": 0,
                "elapsed_s": 0, "output_mb": 0, "path": "—",
                "status": f"ERRO ({motivo})"}

    def _print_banner(self) -> None:
        console.print(Panel(
            "[bold white]FEATURE ENGINEERING v3.0 — NORDESTE | CENÁRIOS A e B[/]\n"
            "[dim]ANTI-LEAKAGE | SEM LAGS/ROLLING | FIX RAM Colab | DOCUMENTAÇÃO INTEGRADA[/]\n\n"
            f"[dim]Input  (Passo 7): {self.input_dir}[/]\n"
            f"[dim]Output (Passo 8): {self.save_dir}[/]\n\n"
            f"[dim]Estratégias: {', '.join(self.strategies)}[/]\n\n"
            "[success]✔  TOP_N_FEATURES 40→20 (C(40,2)=780 → C(20,2)=190 interações)[/]\n"
            "[success]✔  Grupo 5 sem pd.concat — inserção direta column-by-column[/]\n"
            "[success]✔  gc.collect() entre todos os grupos com _log_mem()[/]\n"
            "[success]✔  DATA dropada antes do Grupo 5 (libera RAM)[/]\n"
            "[success]✔  Documentação integrada ao pipeline (FeatureDocumentator)[/]",
            title="[header]  TCC PLD HORÁRIO — PASSO 8: FEATURE ENGINEERING v3.0  [/]",
            border_style="blue", padding=(1, 4),
        ))

    def _print_final_report(self, elapsed_total: float) -> None:
        """Exibe tabela consolidada de todos os splits processados e painel de encerramento."""
        t = Table(
            title="RELATÓRIO CONSOLIDADO — PASSO 8: FEATURE ENGINEERING v3.0",
            border_style="blue", box=box.DOUBLE_EDGE, show_lines=True, min_width=160,
        )
        t.add_column("Cenário",    style="bold",      justify="left",   min_width=28)
        t.add_column("Split",      style="bold",      justify="center", min_width=8)
        t.add_column("Estratégia", style="bold cyan", justify="left",   min_width=22)
        t.add_column("TARGET",     style="dim",       justify="left",   min_width=35)
        t.add_column("Registros",  style="metric",    justify="right",  min_width=12)
        t.add_column("Colunas",    style="success",   justify="right",  min_width=10)
        t.add_column("Saída (MB)", style="success",   justify="right",  min_width=10)
        t.add_column("Tempo (s)",  style="white",     justify="right",  min_width=9)
        t.add_column("Status",     style="bold",      justify="center", min_width=7)

        for r in self._all_results:
            is_a   = "todos" in r.get("cenario", "")
            cor_c  = "cenario_a" if is_a else "cenario_b"
            cor_s  = "treino"    if r["split"] == "TREINO" else "teste"
            status = "[green]OK[/]" if r["status"] == "OK" else f"[red]{r['status']}[/]"
            t.add_row(
                f"[{cor_c}]{r.get('cenario','—')}[/{cor_c}]",
                f"[{cor_s}]{r['split']}[/{cor_s}]",
                r.get("strategy","—"), r.get("target_col","—"),
                f"{r['rows']:,}", f"{r['cols_out']:,}",
                f"{r['output_mb']}", f"{r['elapsed_s']}", status,
            )
        console.print(t)

        n_ok  = sum(1 for r in self._all_results if r["status"] == "OK")
        n_err = len(self._all_results) - n_ok
        console.print(Panel(
            f"[success]PIPELINE COMPLETO EM {elapsed_total:.2f}s[/]\n\n"
            f"   [dim]Cenários    : {len(self.scenarios)}[/]\n"
            f"   [dim]Estratégias : {len(self.strategies)}[/]\n"
            f"   [dim]Splits      : TREINO + TESTE[/]\n"
            f"   [dim]Arquivos OK : {n_ok}  |  Erros: {n_err}[/]\n"
            f"   [dim]Output base : {self.save_dir}[/]",
            title="[header] RESUMO FINAL — PASSO 8 [/]",
            border_style="green", padding=(1, 4),
        ))

    def _salvar_relatorio_cenario(self, results: List[Dict], dir_out: Path) -> None:
        if not results:
            return
        dir_out.mkdir(parents=True, exist_ok=True)
        path = dir_out / "relatorio_feature_engineering.csv"
        pd.DataFrame(results).to_csv(path, index=False, encoding="utf-8-sig")
        console.print(f"   [success]✔  Relatório do cenário salvo:[/] [dim]{path}[/]\n")

    def _salvar_relatorio_consolidado(self) -> None:
        path = self.save_dir / "relatorio_consolidado_feature_engineering.csv"
        pd.DataFrame(self._all_results).to_csv(path, index=False, encoding="utf-8-sig")
        console.print(f"   [success]✔  Relatório consolidado salvo:[/] [dim]{path}[/]\n")


# ==============================================================================
# ▶️ ENTRY POINT
# ==============================================================================
if __name__ == "__main__":
    engine = FeatureEngineeringEngine(
        input_dir     = INPUT_DIR,
        save_dir      = SAVE_DIR,
        scenarios     = SCENARIOS,
        strategies    = STRATEGIES,
        reload_params = False,
    )
    engine.run()